# Fase 3 - Camada Gold: perguntas de negócio e gráficos

Pipeline Medallion (Raw → Silver → **Gold**) sobre os dados de **Viagens a Serviço**
do Portal da Transparência (6 meses de 2025).

Este notebook responde às 7 perguntas de negócio do projeto:

| Parte | Pergunta | Fonte |
|---|---|---|
| A | 4. Tipo de pagamento com maior valor médio | SQL direto na Silver |
| A | 5. Meio de transporte mais usado nos trechos | SQL direto na Silver |
| A | 6. UF de destino em mais trechos | SQL direto na Silver |
| B | 1. 5 órgãos com maior custo total | tabela/VIEW Gold |
| B | 2. 3 destinos com maior custo médio por viagem | tabela/VIEW Gold |
| B | 3. Viagem de maior duração e seu custo total | tabela/VIEW Gold |
| B | 7. Órgão que pagou mais no total | tabela/VIEW Gold |

**Regras de negócio (definidas na Fase 2):**

- `valor_total = valor_diarias + valor_passagens + valor_outros_gastos - valor_devolucao`
- `duracao_dias = data_fim - data_inicio + 1` (conta o dia inicial e o final)
- Pagamentos e passagens 100% idênticos foram tratados como duplicidade e mantidos uma única vez.

**Pré-requisito:** rodar `0_criar_banco.sql`, `1_extrair.py` e `2_transformar.py`, nessa ordem.

## 1. Preparação: bibliotecas, conexão e funções auxiliares

In [ ]:
from decimal import Decimal

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from banco import conectar, executar

In [ ]:
# Cores do projeto (paleta validada para daltonismo e contraste).
COR_PRINCIPAL = "#2a78d6"   # azul: série principal
COR_SECUNDARIA = "#eb6834"  # laranja: segunda série
COR_TEXTO = "#0b0b0b"
COR_TEXTO_SUAVE = "#52514e"
COR_GRADE = "#e3e2de"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TEXTO_SUAVE,
    "axes.labelcolor": COR_TEXTO_SUAVE,
    "xtick.color": COR_TEXTO_SUAVE,
    "ytick.color": COR_TEXTO_SUAVE,
    "axes.titlecolor": COR_TEXTO,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.titlelocation": "left",
    "font.size": 10,
})

# Guarda a resposta de cada pergunta para o resumo final.
respostas = {}

In [ ]:
conexao = conectar()


def consultar(sql):
    """Executa um SELECT e devolve o resultado como DataFrame."""
    cursor = conexao.cursor()
    cursor.execute(sql)
    colunas = [coluna[0] for coluna in cursor.description]
    linhas = cursor.fetchall()
    cursor.close()
    tabela = pd.DataFrame(linhas, columns=colunas)
    for coluna in tabela.columns:
        if tabela[coluna].map(lambda v: isinstance(v, Decimal)).any():
            tabela[coluna] = tabela[coluna].astype(float)
    return tabela


def valor_unico(sql):
    """Executa um SELECT que devolve um único valor e retorna esse valor."""
    return consultar(sql).iloc[0, 0]


def moeda(valor):
    """Formata número como moeda brasileira: 1234.5 -> R$ 1.234,50."""
    texto = f"{valor:,.2f}".replace(",", "X").replace(".", ",")
    return "R$ " + texto.replace("X", ".")


def moeda_eixo(valor):
    """Formato curto para os eixos: 2500000 -> R$ 2,5 mi; 8000 -> R$ 8.000."""
    if abs(valor) >= 1_000_000:
        return "R$ " + f"{valor / 1_000_000:.1f}".replace(".", ",") + " mi"
    return "R$ " + inteiro(valor)


def inteiro(valor):
    """Formata inteiro com separador de milhar: 1234567 -> 1.234.567."""
    return f"{int(valor):,}".replace(",", ".")


def abreviar(texto, limite=38):
    """Encurta textos longos para caberem nos eixos dos gráficos."""
    texto = str(texto)
    return texto if len(texto) <= limite else texto[:limite - 3] + "..."


def mostrar(tabela, colunas_moeda=(), colunas_inteiro=()):
    """Exibe o DataFrame com valores formatados (sem alterar o original)."""
    copia = tabela.copy()
    for coluna in colunas_moeda:
        copia[coluna] = copia[coluna].map(moeda)
    for coluna in colunas_inteiro:
        copia[coluna] = copia[coluna].map(inteiro)
    display(copia.reset_index(drop=True))

In [ ]:
def barras_horizontais(rotulos, valores, titulo, eixo_x, eixo_y, formatar,
                       formatar_eixo=None):
    """
    Gráfico de barras horizontais (maior valor no topo) com o valor
    escrito na ponta de cada barra. Uma única série, uma única cor.
    """
    fig, eixo = plt.subplots(figsize=(9, 0.55 * len(rotulos) + 1.6))
    posicoes = range(len(rotulos))
    barras = eixo.barh(posicoes, valores, height=0.55, color=COR_PRINCIPAL)
    eixo.set_yticks(list(posicoes))
    eixo.set_yticklabels([abreviar(r) for r in rotulos])
    eixo.invert_yaxis()
    eixo.set_title(titulo)
    eixo.set_xlabel(eixo_x)
    eixo.set_ylabel(eixo_y)
    eixo.xaxis.grid(True, color=COR_GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    formatar_eixo = formatar_eixo or formatar
    eixo.xaxis.set_major_formatter(lambda v, _: formatar_eixo(v))
    limite = max(valores) * 1.22
    eixo.set_xlim(0, limite)
    for barra, valor in zip(barras, valores):
        meio_barra = barra.get_y() + barra.get_height() / 2
        eixo.text(valor + limite * 0.01, meio_barra, formatar(valor),
                  va="center", color=COR_TEXTO, fontsize=9)
    fig.tight_layout()
    plt.show()

In [ ]:
# Teste rápido de conexão e do conteúdo da Silver.
contagens = consultar("""
    SELECT 'silver_viagem' AS tabela, COUNT(*) AS linhas FROM silver_viagem
    UNION ALL SELECT 'silver_pagamento', COUNT(*) FROM silver_pagamento
    UNION ALL SELECT 'silver_passagem', COUNT(*) FROM silver_passagem
    UNION ALL SELECT 'silver_trecho', COUNT(*) FROM silver_trecho
""")
mostrar(contagens, colunas_inteiro=["linhas"])